In [171]:
#------------------------------------------------ Import Lib ----------------------------------------
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from time import sleep
import os
import requests
import re

    

In [172]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'TT CBTT' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running TT CBTT Web Scraping Tool v.2.0


In [205]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   


regdict={
    # 'TT CBTT 1': 'https://www.central-bank.org.tt/core-functions/financial-stability/banking-sector/',
    # 'TT CBTT 2': 'https://www.central-bank.org.tt/core-functions/financial-stability/insurance-sector/',
    # 'TT CBTT 3': 'https://www.central-bank.org.tt/core-functions/financial-stability/pensions-sector/',
     'TT CBTT 4': 'https://www.central-bank.org.tt/core-functions/financial-stability/bureau-de-change/',
    }


Typology={

        regulatorName+' 1': 'Regulated Financial Institutions',
        regulatorName+' 2': 'Insurance Sector',
        regulatorName+' 3': 'Pension Sector',
        regulatorName+' 4': 'Bureau de Change'

        }
processdate = now.strftime('%Y-%m-%d')



print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))

The current folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\TT CBTT
The temp folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\TT CBTT\tempfolder


In [199]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




In [206]:
# %%


#------------------------------------------------ Begin_ fileName ----------------------------------------

session = requests.Session()


headers = {

    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",

    "Accept-Encoding": "gzip, deflate, br, zstd",

    "Accept-Language": "en-US,en;q=0.9,zh;q=0.8,zh-CN;q=0.7",

    "Cache-Control": "no-cache",

    "Content-Type": "application/x-www-form-urlencoded",

    "Origin": "https://fsr.mfsa.mt",

    "Referer": "https://fsr.mfsa.mt/",

    "Sec-Fetch-Dest": "iframe",

    "Sec-Fetch-Mode": "navigate",

    "Sec-Fetch-Site": "same-origin",

    "Sec-Fetch-User": "?1",

    "Upgrade-Insecure-Requests": "1",

    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36 Edg/139.0.0.0",

   
}



In [ ]:
for index, reg in enumerate(regdict):
    response = session.get(regdict[reg], headers=headers)
    if response.status_code != 200:
        print(f"{reg}: got {response.status_code}, skipping…")
        continue  
    data = response.text  
    soup = BeautifulSoup(data, "html.parser")
    if reg == 'TT CBTT 1':
        section = soup.find('section',id="institutions-licensed")
        lis = section.find_all('li')
        for li in lis:
            name = li.text
            try:
                website = li.find('a')['href']
            except:
                website = ''
            sqldict['Name'].append(name)
            sqldict['Website'].append(website)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'TT CBTT 2':
        table = soup.find(id = 'table_1')
        tbody = table.find('tbody')
        for tr in tbody:
            tds = tr.find_all('td')
            if len(tds)>0:
                if tds[0].text == 'Active':
                    topology = tds[1].text
                    name = tds[-1].text
                    sqldict['Name'].append(name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split(' ')[0]) 
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['ListName'].append(Typology[reg])   
                    sqldict['Typology'].append(topology)   
                    sqldict = bourange_same_length_array(sqldict)         
    elif reg == 'TT CBTT 3':
        table_content = soup.find('div',class_='w-popup-box-content')    
        lis = table_content.text.split('\n')
        for li in lis:
            sqldict['Name'].append(li)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])  
            sqldict = bourange_same_length_array(sqldict)   
    elif reg == 'TT CBTT 4':
        col_containers = soup.find('section', id =  'listing-of-regulated-institutions')
        wpb_wrappers = col_containers.find_all('div',class_='wpb_wrapper')
        
        for wrapper in wpb_wrappers:
            try:
                title = wrapper.find('p',class_='w-text')
                information = title.find_next_sibling()
                sqldict['Name'].append(title.text)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['ListName'].append(Typology[reg])  


                infos = information.find('p').text
                address_ = infos.split('Telephone:')[0].replace('\n',' ')
                #print(address_)
                sqldict['Address_1'].append(address_)

                tell_ = infos.find('Telephone: ')
                fax_  = infos.find('Fax:')
                
                
                email_ = infos.find('Email: ')
                phone_ = infos[tell_+10:email_].strip()
                if 'Fax:' in phone_:
                    end_ = phone_.find('Fax:')
                    sqldict['Phone'].append(phone_[:end_].strip())

                else:
                    sqldict['Phone'].append(phone_)

                if len(infos[email_:]) >1:
                    email_text = infos[email_:]
                    sqldict['Email'].append(email_text[6:].strip())
                else:
                    sqldict['Email'].append('')
                sqldict = bourange_same_length_array(sqldict)   
            except:
                pass

In [208]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 5 values.
Key 'priority' has 5 values.
Key 'ListLabel' has 5 values.
Key 'Typology' has 5 values.
Key 'EntryType' has 5 values.
Key 'Name' has 5 values.
Key 'InternalID_1' has 5 values.
Key 'InternalID_1_type' has 5 values.
Key 'InternalID_2' has 5 values.
Key 'InternalID_2_type' has 5 values.
Key 'InternalID_3' has 5 values.
Key 'InternalID_3_type' has 5 values.
Key 'CoType' has 5 values.
Key 'License_Type' has 5 values.
Key 'Address_1' has 5 values.
Key 'Address_2' has 5 values.
Key 'City' has 5 values.
Key 'Zip' has 5 values.
Key 'Cntry' has 5 values.
Key 'Phone' has 5 values.
Key 'Fax' has 5 values.
Key 'Website' has 5 values.
Key 'Email' has 5 values.
Key 'RegulationType' has 5 values.
Key 'RegulationTypeCode' has 5 values.
Key 'RegulationDate' has 5 values.
Key 'CancellationDate' has 5 values.
Key 'RegCtry' has 5 values.
Key 'RegCode' has 5 values.
Key 'ListCode' has 5 values.
Key 'ListLanguage' has 5 values.
Key 'ListValidityDate' has 5 values.
Key 'ListName' has

In [209]:

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_5632\19075021.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [210]:
df.to_csv('tt_cbtt_list4_ver2.csv')

In [188]:
import requests
from http.cookies import SimpleCookie

session = requests.Session()

# ── cookies copied from DevTools ──
cookie_header = (
    "COOKIE_SUPPORT=true; _gid=GA1.3.866951197.1761040822; "
    "_ga=GA1.1.1084683801.1761040781; __cf_bm=3mNULpFCN5PMtyi2JQIIpT0.a3Y831YPBz7aoieW8HA-1761054551-1.0.1.1-jNsXDuD9r2lFnHu_EYXa8O0gKGlMehmHTMbRRWtfWjFOGoI5rZnz2KXn2PDg9lHlegNUV31qslfmn81UlQnVTxorEJdkdVA6On2wnDz5zMI; "
    "JSESSIONID=2E8F57212D652BF51409EF3C0C6BD69C; _ga_ZB05G24HQP=GS2.1.s1761054559$o1$g0$t1761054559$j60$l0$h44446931; "
    "_ga_44TNG0HT9N=GS2.1.s1761054598$o1$g0$t1761054598$j60$l0$h0; _ga=GA1.1.291386795.1761054560; "
    "cf_clearance=MlsachelOkN7ZkxPQObK3yqj3uroUmcyoJFTQjvjxH4-1761054604-1.2.1.1-eZ0nZRiSw4by.gz2WEqOsAjTosuCKNgwl.pp.KBPqnIh9FlkxQ2FdVzhUKKyUvzz_zw2NVY3ral1ZIkdk_L_4F41flWaVJHc0JibvubyfvwIgRyWuiJIMhqHVue4wKDMulCgpBywLHmTbiR3i.kLrbRLSCgcDLpPciaqP_jtaOZP4mmfvoNbhFlhEKrD_ELBy_cKa_Z0Aajbl_GDQptAEr8mpCcjHW2kwrXrgEWX_xk; "
    "__cf_bm=iKY3m8ea18x.wM99cXEIh7atCcqbYGShNQmaI4joR0U-1761054604.0105586-1.0.1.1-6BgrepFNYEz.irHyjTYbZVb8GYtDFeONplN.UmNjAoA9Ml3lHZ4mSsACEaVmEs_Apqi_muciVfWERI4FJqtJIJ8HrSg6WtCcIl.eiEQZvCYCzG_PN2UhdSI2A2HSUS37; "
    "_ga=GA1.3.1084683801.1761040781; _ga_CDHDQZB9ET=GS2.3.s1761054644$o2$g0$t1761054644$j60$l0$h0"
)

cookie_jar = SimpleCookie()
cookie_jar.load(cookie_header)
session.cookies.update({key: morsel.value for key, morsel in cookie_jar.items()})

# ── browser-like headers ──
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36 Edg/141.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,zh;q=0.8,zh-CN;q=0.7",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Referer": "https://www.sib.gob.gt/web/sib/entisup",
    "Origin": "https://www.sib.gob.gt",
})

# ── initial page request ──
page_url = "https://www.sib.gob.gt/web/sib/entisup"
response = session.get(page_url)
print("page status:", response.status_code)

# ── example data endpoint (replace with the actual XHR URL you see in DevTools) ──
# data_url = "https://www.sib.gob.gt/some/xhr/endpoint"
# params = {"whatever": "values matching the XHR request"}
# data_response = session.get(data_url, params=params)
# print("data status:", data_response.status_code)
# print(data_response.json())  # or .text depending on the payload


page status: 200


In [192]:
data = response.text  
soup = BeautifulSoup(data, "html.parser")

In [ ]:
soup.find('li',class_='selected')

SyntaxError: incomplete input (3169921271.py, line 1)